In [1]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from datetime import datetime, UTC
import json

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
ROOT = Path("..").resolve()
DATASETS = {
    "train": ROOT / "data/static/v1/scenario_parameter_records_seeded_train.json",
    "val": ROOT / "data/static/v1/scenario_parameter_records_seeded_val.json",
    "holdout": ROOT / "data/static/v1/scenario_parameter_records_seeded_holdout.json",
}
NUMERIC_COLUMNS = [
    "base_spread_prob",
    "wind_strength",
    "spread_rate_1h_m",
    "spread_score",
    "weather_score",
    "cffdrs_dryness_score",
    "size_factor",
    "fire_type_factor",
    "fuel_factor",
    "rain_factor",
]

In [2]:
def load_records(path: Path, split_name: str) -> pd.DataFrame:
    payload = json.loads(path.read_text())
    records = payload.get("records", []) if isinstance(payload, dict) else payload
    frame = pd.DataFrame(records)
    frame["split_source"] = split_name
    return frame

frames = [load_records(path, split) for split, path in DATASETS.items()]
df = pd.concat(frames, ignore_index=True)
df.head()

,record_id,fire_id,source,province,year,split,base_spread_prob,severity_bucket,wind_direction,wind_strength,spread_rate_1h_m,spread_score,weather_score,cffdrs_dryness_score,size_factor,fire_type_factor,fuel_factor,rain_factor,observed_spread_rate_m_min,assessment_hectares,fire_type,fuel_type,record_quality_flag,ignition_seed,layout_seed,split_source
0,AB-2008-MWF074__20080703,AB-2008-MWF074,AB_HISTORICAL_WILDFIRE,AB,2008,train,0.1792,high,S,0.1625,6000.0,0.7736,0.4535,0.0,0.9500,1.00,1.12,1.0,100.0,0.50,surface,C2,measured,14443855870861270751,1.127873e+19,train
1,AB-2018-SWF158__20180901,AB-2018-SWF158,AB_HISTORICAL_WILDFIRE,AB,2018,train,0.1571,medium,NW,0.1625,4260.0,0.6504,0.2518,0.0,0.9500,1.00,1.00,1.0,71.0,0.01,surface,O1b,measured,10198127025334700556,1.445778e+19,train
2,AB-2012-MWF047__20120710,AB-2012-MWF047,AB_HISTORICAL_WILDFIRE,AB,2012,train,0.1797,high,W,0.1625,4200.0,0.7759,0.4640,0.0,0.9500,1.00,1.12,1.0,70.0,0.30,surface,C2,measured,8000671072565187824,8.493056e+17,train
3,AB-2014-GWF044__20140715,AB-2014-GWF044,AB_HISTORICAL_WILDFIRE,AB,2014,train,0.2200,high,SW,0.2625,3900.0,1.0000,0.5501,0.0,1.0250,1.18,1.12,1.0,65.0,1000.00,crown,C3,measured,15167517599069789484,1.618321e+19,train
4,AB-2017-CWF254__20171017,AB-2017-CWF254,AB_HISTORICAL_WILDFIRE,AB,2017,train,0.2194,high,W,0.6000,3600.0,0.9966,0.7679,0.0,0.9507,1.18,1.12,1.0,60.0,10.00,crown,C3,measured,2787351251254423658,1.730736e+19,train


In [3]:
summary = {
    "rows_total": int(len(df)),
    "rows_by_split_source": df["split_source"].value_counts(dropna=False).to_dict(),
    "rows_by_split_field": df["split"].value_counts(dropna=False).to_dict(),
    "unique_record_id": int(df["record_id"].nunique(dropna=True)),
    "duplicate_record_id_count": int(df.duplicated(subset=["record_id"]).sum()),
    "duplicate_fire_id_count": int(df.duplicated(subset=["fire_id"]).sum()),
}
summary

{'rows_total': 19246,
 'rows_by_split_source': {'train': 18252, 'val': 993, 'holdout': 1},
 'rows_by_split_field': {'train': 18252, 'val': 993, 'holdout': 1},
 'unique_record_id': 19246,
 'duplicate_record_id_count': 0,
 'duplicate_fire_id_count': 0}

In [4]:
required = [
    "record_id",
    "split",
    "base_spread_prob",
    "severity_bucket",
    "wind_direction",
    "wind_strength",
    "ignition_seed",
    "layout_seed",
]
null_counts = df[required].isna().sum().sort_values(ascending=False)
null_counts

record_id           0
split               0
base_spread_prob    0
severity_bucket     0
wind_direction      0
wind_strength       0
ignition_seed       0
layout_seed         0
dtype: int64

In [5]:
print("All columns:")
print(df.columns.tolist())

const_cols = []
for col in NUMERIC_COLUMNS:
    unique_vals = df[col].nunique()
    if unique_vals <= 1:
        print(f"\nConstant column: {col}")
        const_cols.append(col)

print("\nNumeric cols:")
valid_numeric_cols = [col for col in NUMERIC_COLUMNS if col in df.columns and col not in const_cols]
print(valid_numeric_cols)

All columns:
['record_id', 'fire_id', 'source', 'province', 'year', 'split', 'base_spread_prob', 'severity_bucket', 'wind_direction', 'wind_strength', 'spread_rate_1h_m', 'spread_score', 'weather_score', 'cffdrs_dryness_score', 'size_factor', 'fire_type_factor', 'fuel_factor', 'rain_factor', 'observed_spread_rate_m_min', 'assessment_hectares', 'fire_type', 'fuel_type', 'record_quality_flag', 'ignition_seed', 'layout_seed', 'split_source']

Constant column: cffdrs_dryness_score

Numeric cols:
['base_spread_prob', 'wind_strength', 'spread_rate_1h_m', 'spread_score', 'weather_score', 'size_factor', 'fire_type_factor', 'fuel_factor', 'rain_factor']


In [6]:
cols = [
    "record_id",
    "fire_id",
    "split",
    "split_source",
    "severity_bucket",
    "spread_rate_1h_m",
    "base_spread_prob",
    "spread_score",
    "wind_strength",
    "weather_score"
]

df.loc[df["spread_rate_1h_m"] < 0, cols]

,record_id,fire_id,split,split_source,severity_bucket,spread_rate_1h_m,base_spread_prob,spread_score,wind_strength,weather_score
18245,AB-2014-RWF022__20140524,AB-2014-RWF022,train,train,low,-60.0,0.0452,0.0290,0.1125,0.1618
18246,AB-2017-MWF091__20170916,AB-2017-MWF091,train,train,low,-60.0,0.0606,0.1147,0.3500,0.5736
18247,AB-2019-SWF092__20190602,AB-2019-SWF092,train,train,low,-60.0,0.0599,0.1106,0.2875,0.4940
18248,AB-2015-HWF100__20150518,AB-2015-HWF100,train,train,low,-60.0,0.0590,0.1055,0.2250,0.5274
18249,AB-2022-RWF056__20220820,AB-2022-RWF056,train,train,low,-60.0,0.0477,0.0427,0.1625,0.1905
18250,AB-2020-CWF038__20200628,AB-2020-CWF038,train,train,low,-60.0,0.0460,0.0335,0.1250,0.1677
18251,AB-2019-SWF063__20190522,AB-2019-SWF063,train,train,low,-60.0,0.0529,0.0714,0.1625,0.4210


There is no transformation in the data pipeline, which:

- Requires raw FIRE_SPREAD_RATE to be present (otherwise row is dropped)
- Parses FIRE_SPREAD_RATE as float into observed_spread_rate_m_min and writes it unchanged to normalized records
- Drops the row if spread rate cannot be parsed (None)
- Computes spread_rate_1h_m as a derived field: observed_spread_rate_m_min * 60, rounded to 1 decimal

So the negative values came from the Alberta dataset itself. And since the negative semantics are not known, dropping the 7 out of 19246 rows is my best bet.

In [7]:
df = df[df["spread_rate_1h_m"] >= 0].copy()
df = df.drop(columns=["cffdrs_dryness_score"])

In [8]:
# check again 
cols = [
    "record_id",
    "fire_id",
    "split",
    "split_source",
    "severity_bucket",
    "spread_rate_1h_m",
    "base_spread_prob",
    "spread_score",
    "wind_strength",
    "weather_score"
]

df.loc[df["spread_rate_1h_m"] < 0, cols]

,record_id,fire_id,split,split_source,severity_bucket,spread_rate_1h_m,base_spread_prob,spread_score,wind_strength,weather_score


In [9]:
# check if cffdrs col was dropped
print(df.columns)

Index(['record_id', 'fire_id', 'source', 'province', 'year', 'split',
       'base_spread_prob', 'severity_bucket', 'wind_direction',
       'wind_strength', 'spread_rate_1h_m', 'spread_score', 'weather_score',
       'size_factor', 'fire_type_factor', 'fuel_factor', 'rain_factor',
       'observed_spread_rate_m_min', 'assessment_hectares', 'fire_type',
       'fuel_type', 'record_quality_flag', 'ignition_seed', 'layout_seed',
       'split_source'],
      dtype='str')


In [10]:
# check split vs split_name for matching record_count 
for split_name in ["train", "val", "holdout"]:
    n_df = (df["split"] == split_name).sum()
    print(split_name, n_df)

train 18245
val 993
holdout 1


In [11]:
OUT_DIR = ROOT / "data/static/v2"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Sanity check: helper split_source must match canonical split
if "split_source" in df.columns and "split" in df.columns:
    mismatch_mask = df["split_source"] != df["split"]
    if mismatch_mask.any():
        mismatch_rows = df.loc[mismatch_mask, ["record_id", "split", "split_source"]]
        raise ValueError(
            f"split_source does not match split for {mismatch_mask.sum()} rows.\n"
            f"Example mismatches:\n{mismatch_rows.head(10)}"
        )

# Make sure seed columns are true ints
for col in ["ignition_seed", "layout_seed", "year"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("UInt64")

preferred_cols = [
    "record_id", "fire_id", "source", "province", "year", "split",
    "base_spread_prob", "severity_bucket", "wind_direction", "wind_strength",
    "spread_rate_1h_m", "spread_score", "weather_score",
    "size_factor", "fire_type_factor", "fuel_factor", "rain_factor",
    "observed_spread_rate_m_min", "assessment_hectares",
    "fire_type", "fuel_type", "record_quality_flag",
    "ignition_seed", "layout_seed",
]

# Keep schema fields, drop temporary helper split_source
ordered_cols = [c for c in preferred_cols if c in df.columns] + [
    c for c in df.columns if c not in preferred_cols and c != "split_source"
]

def _to_py_records(frame: pd.DataFrame) -> list[dict]:
    records = []
    for rec in frame[ordered_cols].to_dict(orient="records"):
        out = {}
        for k, v in rec.items():
            if pd.isna(v):
                out[k] = None
            elif isinstance(v, (np.integer,)):
                out[k] = int(v)
            elif isinstance(v, (np.floating, float)):
                out[k] = None if not np.isfinite(v) else float(v)
            elif isinstance(v, (np.bool_,)):
                out[k] = bool(v)
            else:
                out[k] = v
        records.append(out)
    return records

# Write per-split seeded files using canonical split field
for split_name in ["train", "val", "holdout"]:
    split_df = df[df["split"] == split_name].copy()
    records = _to_py_records(split_df)

    payload = {
        "schema_version": 3,
        "generated_at": datetime.now(UTC).isoformat(),
        "split": split_name,
        "record_count": len(records),
        "records": records,
    }

    out_path = OUT_DIR / f"scenario_parameter_records_seeded_{split_name}.json"
    out_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"Wrote {len(records):>6} records -> {out_path}")

# Optional combined seeded file
all_records = _to_py_records(df.copy())
combined_payload = {
    "schema_version": 3,
    "generated_at": datetime.now(UTC).isoformat(),
    "record_count": len(all_records),
    "records": all_records,
}
combined_path = OUT_DIR / "scenario_parameter_records_seeded.json"
combined_path.write_text(json.dumps(combined_payload, indent=2), encoding="utf-8")
print(f"Wrote {len(all_records):>6} records -> {combined_path}")

Wrote  18245 records -> /home/makerspace/thomson/firebot-eval/data/static/v2/scenario_parameter_records_seeded_train.json
Wrote    993 records -> /home/makerspace/thomson/firebot-eval/data/static/v2/scenario_parameter_records_seeded_val.json
Wrote      1 records -> /home/makerspace/thomson/firebot-eval/data/static/v2/scenario_parameter_records_seeded_holdout.json
Wrote  19239 records -> /home/makerspace/thomson/firebot-eval/data/static/v2/scenario_parameter_records_seeded.json
